# Customer Segmentation & Behavioral Analysis

Self-initiated Data Analyst portfolio project using Python, Pandas, RFM analysis, SQL, and Power BI.


## 1. Project Objective

Analyze customer transaction data, calculate Recency, Frequency, and Monetary (RFM) metrics, create customer segments, and generate business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## 2. Load Cleaned Data

The cleaned Excel dataset contains 1,500 transaction records after duplicate removal.

In [ ]:
# Update this path when running the notebook on another computer.
file_path = r"Project2_Customer_Segmentation_Cleaned.xlsx"
df = pd.read_excel(file_path)

df.head()

In [ ]:
df.shape


## 3. Data Quality Checks

In [ ]:
df.info()

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 4. Prepare Dates and Customer-Level Summary

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

customer_summary = (
    df.groupby("Customer_ID")
      .agg(
          Customer_Name=("Customer_Name", "first"),
          Last_Purchase_Date=("Order_Date", "max"),
          Total_Orders=("Order_ID", "nunique"),
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Profit", "sum"),
          Total_Quantity=("Quantity", "sum")
      )
      .reset_index()
)

customer_summary.head()

## 5. RFM Analysis

- **Recency:** Days since the customer's latest purchase.
- **Frequency:** Number of unique orders.
- **Monetary:** Total customer sales.

In [ ]:
analysis_date = df["Order_Date"].max() + pd.Timedelta(days=1)

customer_summary["Recency"] = (
    analysis_date - customer_summary["Last_Purchase_Date"]
).dt.days
customer_summary["Frequency"] = customer_summary["Total_Orders"]
customer_summary["Monetary"] = customer_summary["Total_Sales"]

customer_summary[["Customer_ID", "Recency", "Frequency", "Monetary"]].head()

## 6. RFM Scoring

Customers are scored from 1 to 5. For Recency, a higher score represents more recent activity.

In [ ]:
customer_summary["R_Score"] = pd.qcut(
    customer_summary["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

customer_summary["F_Score"] = pd.qcut(
    customer_summary["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

customer_summary["M_Score"] = pd.qcut(
    customer_summary["Monetary"],
    5,
    labels=[1, 2, 3, 4, 5]
)

customer_summary["RFM_Score"] = (
    customer_summary["R_Score"].astype(int).astype(str)
    + customer_summary["F_Score"].astype(int).astype(str)
    + customer_summary["M_Score"].astype(int).astype(str)
)

customer_summary[["Customer_ID", "R_Score", "F_Score", "M_Score", "RFM_Score"]].head()

## 7. Customer Segmentation

In [ ]:
def assign_segment(row):
    if row["R_Score"] >= 4 and row["F_Score"] >= 4 and row["M_Score"] >= 4:
        return "Champions"
    elif row["R_Score"] >= 3 and row["F_Score"] >= 3:
        return "Loyal Customers"
    elif row["R_Score"] >= 4 and row["F_Score"] <= 2:
        return "New / Promising"
    elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
        return "At Risk"
    else:
        return "Low Engagement"

customer_summary["Customer_Segment"] = customer_summary.apply(assign_segment, axis=1)

customer_summary["Customer_Segment"].value_counts()

## 8. Segment-Level Summary

In [ ]:
segment_summary = (
    customer_summary.groupby("Customer_Segment")
    .agg(
        Customers=("Customer_ID", "nunique"),
        Total_Sales=("Total_Sales", "sum"),
        Total_Profit=("Total_Profit", "sum"),
        Average_Customer_Sales=("Total_Sales", "mean")
    )
    .reset_index()
    .sort_values("Total_Sales", ascending=False)
)

segment_summary

## 9. Visualize Customer Segments

In [ ]:
segment_counts = customer_summary["Customer_Segment"].value_counts()

plt.figure(figsize=(8, 5))
segment_counts.plot(kind="bar")
plt.title("Customer Segment Distribution")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 10. Recency vs Frequency

In [ ]:
plt.figure(figsize=(8, 5))
for segment, group in customer_summary.groupby("Customer_Segment"):
    plt.scatter(group["Recency"], group["Frequency"], label=segment, alpha=0.7)

plt.title("Recency vs Frequency by Customer Segment")
plt.xlabel("Recency (Days)")
plt.ylabel("Frequency (Orders)")
plt.legend()
plt.tight_layout()
plt.show()

## 11. At-Risk Customers

In [ ]:
at_risk_customers = (
    customer_summary[customer_summary["Customer_Segment"] == "At Risk"]
    .sort_values("Total_Sales", ascending=False)
)

at_risk_customers[
    ["Customer_ID", "Customer_Name", "Total_Sales", "Total_Orders", "Recency", "Frequency", "Monetary"]
].head(20)

## 12. Export Customer-Level Analysis

The resulting customer-level table can be used for SQL analysis and Power BI reporting.

In [ ]:
output_file = "Project2_Customer_RFM_Analysis.xlsx"
customer_summary.to_excel(output_file, index=False)
print(f"Saved: {output_file}")

## 13. Project Conclusion

RFM analysis provides a structured way to understand customer engagement and value. The resulting customer segments can support targeted engagement, retention, and customer-value analysis in the Power BI dashboard.

**Portfolio note:** This is a self-initiated project using synthetic/realistic transaction data and is not based on confidential client data.